# Tutorial LangChain (Básico → Intermedio)

Este notebook te guía paso a paso para aprender LangChain con `langchain==1.2.10` y Ollama.

## Objetivos
- Entender los bloques principales de LangChain
- Construir cadenas simples con LCEL
- Trabajar con prompts, modelos y parsers
- Crear un mini sistema RAG local (nivel intermedio)

## 0) Preparación

Si ya tienes todo instalado, puedes saltar esta celda.

In [1]:
#%pip install -q langchain langchain-core langchain-community langchain-ollama chromadb tiktoken

In [2]:
import langchain
print('LangChain version:', langchain.__version__)

LangChain version: 1.2.10


## 1) Tu primer bloque: Prompt + Modelo + Parser

En LangChain, una app LLM suele tener:
- PromptTemplate (instrucción)
- Modelo (LLM o chat model)
- Parser (cómo quieres la salida)

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser


In [5]:
prompt = ChatPromptTemplate.from_template(
    'Explain in simple words this {topic} and include examples.'
)

llm = ChatOllama(model='tinyllama', temperature=0)

parser = StrOutputParser()


In [6]:
chain = prompt | llm | parser
respuesta = chain.invoke({'topic': 'emg signal processing'})
print(respuesta)

Emg signal processing is a process of analyzing and interpreting electrical signals generated by muscles during physical activity. The signals are typically generated by the muscles during exercise, and they are used to measure the intensity, duration, and type of exercise performed.

Here are some steps involved in emg signal processing:

1. Electrode placement: The electrodes are placed on the muscles to measure the electrical signals generated by the muscles. The placement of the electrodes can vary depending on the type of exercise being performed.

2. Data acquisition: The data is collected and stored in a digital format.

3. Signal processing: The signals are processed to extract the relevant information. This can involve filtering, amplification, and analysis.

4. Analysis: The data is analyzed to determine the intensity, duration, and type of exercise performed.

Examples of emg signal processing include:

1. Heart rate monitoring: Heart rate is a useful indicator of exercise i

In [7]:
chain = prompt | llm 
respuesta = chain.invoke({'topic': 'LangChain'})
print(respuesta.content)

LangChain is a powerful AI assistant that can help you with your language needs. It is a machine learning model that can translate text from one language to another. Here's how LangChain works:

1. Input: The first step is to provide LangChain with the text you want to translate.

2. Language Selection: LangChain has multiple language options, including English, Spanish, French, German, and more. Select the language you want to translate to.

3. Input Text: LangChain will now analyze the text you provided and determine the best translation for you.

4. Translation: Once LangChain has identified the best translation, it will provide you with the translated text.

Here are some examples of how LangChain can help you:

1. Translation of a book: If you're reading a book in English and want to translate it to Spanish, LangChain can help.

2. Translation of a website: If you're browsing a website in Spanish and want to translate it to English, LangChain can help.

3. Translation of a documen

## 2) LCEL: componer cadenas

LCEL (LangChain Expression Language) te permite conectar bloques con `|`.

De forma general, una LCEL (LangChain Expression Language) es una tubería de pasos conectados con |, donde cada paso transforma datos para el siguiente.

Estructura típica:
```
1. Entrada (str, dict, etc.)
2. Preproceso (opcional, p. ej. RunnableLambda)
3. Prompt (PromptTemplate / ChatPromptTemplate)
3. Modelo (ChatOllama, etc.)
4. Postproceso (StrOutputParser, JSON parser, validación)
```

In [8]:
from langchain_core.runnables import RunnableLambda

normalizar = RunnableLambda(lambda x: {'topic': x.strip().lower()})
chain_lcel = normalizar | prompt | llm | parser

print(chain_lcel.invoke('ecg signal  processing'))

Sure, I'd be happy to explain this ecg signal processing in simple words!

An electrocardiogram (ecg) signal is a graphical representation of the electrical activity of the heart. It is typically displayed as a series of lines or curves, each representing a different electrode (or electrodes) on the heart. The lines or curves are called electrocardiograms (ecgs) and are typically displayed on a monitor or chart.

Here's an example of how an ecg signal might look:

![ECG Signal](https://i.imgur.com/VbZVVVV.png)

In this example, the lines or curves represent the electrical activity of the left atrium (L1), left ventricle (LV), and right atrium (R1). The lines or curves are colored based on the strength of the electrical signal, with darker colors representing stronger signals.

Here are some examples of how ecg signals are processed:

1. Preprocessing: Before processing the ecg signal, it is usually preprocessed to remove any unwanted noise or interference. This can be done using techni

## 3) Salidas estructuradas (JSON simple)

Aquí pedimos JSON y luego lo parseamos con Python.

In [ ]:
import json

prompt_json = ChatPromptTemplate.from_template(
    'Devuelve SOLO JSON válido con claves: d' \
    'efinicion, ejemplo. Tema: {tema}'
)

chain_json = prompt_json | llm | parser


raw = chain_json.invoke({'tema': 'embeddings'})

print('Salida cruda:\n', raw)

try:
    data = json.loads(raw)
    print('\nJSON parseado:', data)
except Exception as e:
    print('\nNo se pudo parsear JSON:', e)

## 4) Nivel intermedio I: recuperación semántica (vector store)

Crearemos documentos locales, embeddings y un retriever.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

docs_base = [
    Document(page_content='LangChain es un framework para construir aplicaciones con LLMs.'),
    Document(page_content='RAG combina recuperación de documentos y generación de texto.'),
    Document(page_content='Los embeddings convierten texto en vectores para medir similitud semántica.'),
    Document(page_content='Un retriever encuentra documentos relevantes para una pregunta.')
]

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
splits = splitter.split_documents(docs_base)

embeddings = OllamaEmbeddings(model='nomic-embed-text')
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

print('Chunks indexados:', len(splits))

In [ ]:
consulta = '¿Para qué sirven los embeddings?'
docs_recuperados = retriever.invoke(consulta)
for i, d in enumerate(docs_recuperados, 1):
    print(f'[{i}]', d.page_content)

## 5) Nivel intermedio II: mini RAG end-to-end

Conectamos retriever + prompt + modelo en una sola cadena.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_template(
    'Responde usando SOLO este contexto:\n{context}\n\nPregunta: {question}'
)

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm
    | parser
)

print(rag_chain.invoke('¿Qué es RAG y qué papel cumple un retriever?'))

## 6) Siguiente nivel (qué aprender después)

Ya estás en nivel intermedio inicial. Aquí tienes una ruta concreta para avanzar de forma práctica.

### 6.1 Evaluación de respuestas (calidad y grounding)
Meta: medir si la respuesta realmente usa el contexto.

### 6.2 Búsqueda híbrida y reranking
Meta: recuperar mejores documentos combinando estrategias.

### 6.3 Agentes y tools
Meta: permitir que el modelo use funciones externas.

### 6.4 Memoria conversacional
Meta: mantener contexto entre turnos.

### 6.5 Orquestación con LangGraph
Meta: pasar de cadenas lineales a flujos con estados y ramas.

### 6.1 Mini evaluación automática de un RAG

Calculamos métricas simples:
- `context_recall`: si recuperamos algo útil
- `grounded_answer`: si la respuesta comparte términos con el contexto

In [ ]:
def simple_overlap_score(a: str, b: str) -> float:
    a_set = set(w.lower().strip('.,:;!?¿¡()[]') for w in a.split() if len(w) > 3)
    b_set = set(w.lower().strip('.,:;!?¿¡()[]') for w in b.split() if len(w) > 3)
    if not a_set or not b_set:
        return 0.0
    return len(a_set & b_set) / max(1, len(a_set))

pregunta_eval = '¿Para qué sirve un retriever en RAG?'
docs_eval = retriever.invoke(pregunta_eval)
respuesta_eval = rag_chain.invoke(pregunta_eval)
contexto_eval = '\n\n'.join(d.page_content for d in docs_eval)

context_recall = 1.0 if len(docs_eval) > 0 else 0.0
grounded_answer = simple_overlap_score(str(respuesta_eval), contexto_eval)

print('Pregunta:', pregunta_eval)
print('Docs recuperados:', len(docs_eval))
print('Context recall:', context_recall)
print('Grounded score (0-1):', round(grounded_answer, 3))
print('Respuesta:\n', respuesta_eval)

### 6.2 Búsqueda híbrida (idea base)

Aquí hacemos una versión simple: combinamos recuperación vectorial con un score por palabra clave.

In [ ]:
def keyword_score(query: str, text: str) -> float:
    q = set(query.lower().split())
    t = set(text.lower().split())
    if not q:
        return 0.0
    return len(q & t) / len(q)

def hybrid_retrieve(query: str, k: int = 2):
    vec_docs = retriever.invoke(query)
    scored = []
    for d in vec_docs:
        ks = keyword_score(query, d.page_content)
        scored.append((ks, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in scored[:k]]

docs_hybrid = hybrid_retrieve('¿Qué rol tienen los embeddings en RAG?')
for i, d in enumerate(docs_hybrid, 1):
    print(f'[Hybrid {i}]', d.page_content)

### 6.3 Primer paso en agentes con tools

Mostramos cómo registrar herramientas (`@tool`) y llamarlas desde Python.

In [ ]:
from langchain_core.tools import tool

@tool
def sumar(a: int, b: int) -> int:
    """Suma dos enteros."""
    return a + b

@tool
def longitud_texto(texto: str) -> int:
    """Devuelve la longitud de un texto."""
    return len(texto)

print('Tool sumar:', sumar.invoke({'a': 7, 'b': 5}))
print('Tool longitud_texto:', longitud_texto.invoke({'texto': 'LangChain'}))